In [56]:
import cobra
import escher
import warnings

In [21]:
model = cobra.io.read_sbml_model("model.xml")

In [22]:
# Save the path to the Escher map (with glycolysis and TCA cycle)
amac_map_path = "escher/MIT1002_glycolysis_and_tca_escher-map.json"

In [23]:
# Set the medium
minimal_media = {
    "EX_cpd00027_e0": 10,  # Glucose_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00013_e0": 1000,  # NH3_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00075_e0": 1000,  # Nitrite_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
# Set the model medium
model.medium = minimal_media

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


In [24]:
model.reactions.EX_cpd00305_e0

Reaction identifier,EX_cpd00305_e0
Name,Exchange for Thiamin [e0]
Memory address,0x7fb9a3999400
Stoichiometry,cpd00305_e0 --> Thiamin [e0] -->
GPR,
Lower bound,-0.0
Upper bound,1000.0


In [25]:
pfba_solution = cobra.flux_analysis.pfba(model)

In [26]:
model.reactions.bio1_biomass.metabolites[model.metabolites.cpd00056_c0]

-0.00160004066244054

In [27]:
0.0016 * 0.915

0.001464

In [28]:
# Export the reaction data so I can use the web version of Escher
import json
with open("pfba_fluxes.json", "w") as f:
    json.dump(pfba_solution.fluxes.to_dict(), f)

In [33]:
# Set TPP production reaction as objective and optimize
model.objective = {model.reactions.rxn00438_c0: 1}
pfba_tpp_solution = cobra.flux_analysis.pfba(model)

In [36]:
pfba_tpp_solution.fluxes["rxn00438_c0"]

0.0014633854938381839

In [44]:
# Get all the metabolites that are taken up from the media (reactions that start with "EX_" and with negative flux)
tpp_media_reqs = pfba_tpp_solution.fluxes[(pfba_tpp_solution.fluxes < 0) & (pfba_tpp_solution.fluxes.index.str.startswith("EX_"))]

In [73]:
# Define Franzi's media
# Based on 2024-02-19_Kratzl_Marine_Pro_Medium.xlsx
marine_broth_wo_yeast_and_peptone = {
    # The definition from my media file
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00001_e0": 1000,  # H2O
    ################
    # Salt Solution
    ################
    # NaCl
    "EX_cpd00971_e0": 1000,  # Na+
    "EX_cpd00099_e0": 1000,  # Cl-
    # MgSO4*7H2O
    "EX_cpd00254_e0": 1000,  # Mg2+
    "EX_cpd00048_e0": 1000,  # Sulfate (O4S)
    # MgCl2
    # Mg2+ already included in MgSO4*7H2O
    # Cl- already included in NaCl
    # KCl
    # Cl- already included in NaCl
    "EX_cpd00205_e0": 1000,  # K+
    # CaCl2
    # Cl- already included in NaCl
    "EX_cpd00063_e0": 1000,  # Ca2+
    # Boric acid
    "EX_cpd09225_e0": 1000,  # Boric acid (H3BO3)
    # NaHCO3
    # Na+ already included in NaCl
    "EX_cpd00242_e0": 1000,  # Biocarbonate (HCO3-)
    # Na2PO4
    # Na+ already included in NaCl
    "EX_cpd00009_e0": 1000,  # Phosphate (HO4P)
    # FeCl3*6H2O
    # Cl- already included in NaCl
    "EX_cpd10516_e0": 1000,  # fe3_e0
    ####################
    # Nitrogen Solution
    ####################
    # NH4Cl
    # Cl- already included in NaCl
    "EX_cpd00013_e0": 1000,  # Ammonia
    # KNO3
    # K+ already included in KCl
    "EX_cpd00209_e0": 1000,  # NO3-
    ###########
    # Vitamins
    ###########
    # Thiamine HCl
    # Cl- already included in NaCl
    # Assuming I do not need to add H
    "EX_cpd00305_e0": 1000,  # Thiamine
    # Biotin
    "EX_cpd00104_e0": 1000,  # Biotin
    # B12 (cyanocobalamin)
    "EX_cpd01826_e0": 1000,  # Cyanocobalamin
    # Folic acid
    "EX_cpd00393_e0": 1000,  # Folate
    # PABA
    "EX_cpd00443_e0": 1000,  # p-Aminobenzoic acid
    # Nicotinic acid (niacin)
    "EX_cpd00133_e0": 1000,  # Nicotinic acid
    # Inositol
    "EX_cpd00121_e0": 1000,  # Inositol
    # Ca Pantothanate
    # Ca2+ already included in CaCl2
    "EX_cpd00644_e0": 1000,  # Pantothenic acid
    # Pyridoxine HCl
    # Cl- already included in NaCl
    "EX_cpd00263_e0": 1000,  # Pyridoxine
    #################
    # Trace Elements
    #################
    # ZnSO4*7H2O
    # Sulfate already included in MgSO4
    "EX_cpd00034_e0": 1000,  # Zn2+
    # CoCl2*6H2O
    # Cl- already included in NaCl
    "EX_cpd00149_e0": 1000,  # Co2+
    # MnCl2*4H2O
    # Cl- already included in NaCl
    "EX_cpd00030_e0": 1000,  # Mn2+
    # Na2MoO4*2H2O
    # Na+ already included in NaCl
    "EX_cpd11574_e0": 1000,  # Molybdate (MoO4)
    # Na2SeO3
    # Na+ already included in NaCl
    "EX_cpd03387_e0": 1000,  # Selenite (O3Se)
    # NiCl2*6H2O
    # Cl- already included in NaCl
    "EX_cpd00244_e0": 1000,  # Ni2+
}

In [74]:
# Check that everything in tpp media reqs is in Franzi's medium
for r in tpp_media_reqs.index:
    if not r in marine_broth_wo_yeast_and_peptone:
        print(f"{r} ({model.reactions.get_by_id(r).name}) is missing from Franzi's medium")

EX_cpd00075_e0 (Exchange for Nitrite [e0]) is missing from Franzi's medium
EX_cpd00058_e0 (Exchange for Cu2+ [e0]) is missing from Franzi's medium
EX_cpd00027_e0 (Exchange for D-Glucose [e0]) is missing from Franzi's medium


In [83]:
# Add missing components to Franzi's medium
test_medium = marine_broth_wo_yeast_and_peptone.copy()
# test_medium["EX_cpd00075_e0"] = 1000
test_medium["EX_cpd00058_e0"] = 1000
test_medium["EX_cpd00027_e0"] = 10

In [53]:
def clean_media(model: cobra.Model, media: dict) -> dict:
    """clean_media
    Removes exchange reactions from the media that are not present in the model

    Parameters
    ----------
    model : cobra.Model
        The model to set the media for.
    media : dict
        A dictionary where the keys are the exchange reactions for the metabolites
        in the media, and the values are the lower bound for the exchange reaction.

    Returns
    -------
    dict
        A dictionary where the keys are the exchange reactions for the metabolites
        in the media, and the values are the lower bound for the exchange reaction
    """
    # Make an empty dictionary for the media
    clean_medium = {}
    # Loop through the media and set the exchange reactions that are present
    for ex_rxn, lb in media.items():
        if ex_rxn in [r.id for r in model.reactions]:
            clean_medium[ex_rxn] = lb
        else:
            warnings.warn(
                "Model does not have the exchange reaction "
                + ex_rxn
                + ", so it was not set in the media."
            )

    # Return the clean medium
    return clean_medium

In [84]:
model.medium = clean_media(model, test_medium)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_21079/3071354421.py:26: UserWarning: Model does not have the exchange reaction EX_cpd09225_e0, so it was not set in the media.
  warnings.warn(
/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_21079/3071354421.py:26: UserWarning: Model does not have the exchange reaction EX_cpd00242_e0, so it was not set in the media.
  warnings.warn(
/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_21079/3071354421.py:26: UserWarning: Model does not have the exchange reaction EX_cpd00104_e0, so it was not set in the media.
  warnings.warn(
/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_21079/3071354421.py:26: UserWarning: Model does not have the exchange reaction EX_cpd01826_e0, so it was not set in the media.
  warnings.warn(
/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_21079/3071354421.py:26: UserWarning: Model does not have the exchange reaction EX_cpd00393_e0, so it was not set in the media.
  

In [85]:
model.objective = "bio1_biomass"
model.optimize()

,fluxes,reduced_costs
rxn02201_c0,0.004341,-3.469447e-16
rxn00351_c0,0.000000,-4.521641e-02
rxn07431_c0,0.000000,0.000000e+00
rxn00836_c0,0.000000,-2.260820e-02
rxn00423_c0,0.000000,-4.521641e-02
...,...,...
EX_cpd00039_e0,0.000000,-1.130410e-01
EX_cpd00054_e0,0.000000,-4.521641e-02
rxn15341_c0,0.000000,1.386552e-18
rxn01519_c0,0.005684,-1.387779e-17
